# Connect to OpenAI

In [ ]:
from openai import OpenAI
from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain.chat_models import init_chat_model
import gradio as gr
from langchain_core.messages import AIMessage



load_dotenv()
client = OpenAI()
embeddings_model = OpenAIEmbeddings()
response_model = init_chat_model("gpt-3.5-turbo-0125", temperature=0)

GENERATE_PROMPT = (
    "You are an assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer the question. "
    "If you don't know the answer, just say that you don't know. "
    "Use three sentences maximum and keep the answer concise.\n"
    "Question: {question} \n"
    "Context: {context}"
)

/Users/sumithra/chatbot/.venv/lib/python3.14/site-packages/langchain_core/_api/deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1
/Users/sumithra/chatbot/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# PDF Loader

In [2]:

def load_and_embed_document():
    pdf_loader = PyPDFLoader("nestle_hr_policy_pdf_2012.pdf")
    pages = pdf_loader.load_and_split()
    content=[]
    for p in pages:
        content.append(p.page_content)
    splitter = RecursiveCharacterTextSplitter(chunk_size=10, chunk_overlap=0, separators=['\n'])
    chunks = splitter.split_text("".join(content))
    #embeddings = embeddings_model.embed_documents(chunks)
    faiss_index = FAISS.from_documents(pages, embeddings_model)
    faiss_index.save_local(".")
    return faiss_index


In [3]:

def retrieve_context(query: str,faiss_index:hash):
    """Retrieve information to help answer a query."""
    retrieved_docs = faiss_index.similarity_search(query, k=2)
    return "\n\n".join([doc.page_content for doc in retrieved_docs])

In [33]:


def generate_answer(question:str, context:str):
    """Generate an answer."""
    prompt = GENERATE_PROMPT.format(question=question, context=context)
    response = response_model.invoke([{"role": "user", "content": prompt}])
    return response


In [41]:
faiss_index = load_and_embed_document()

    #return user_message

def respond(message, history):
    # In newer Gradio, 'message' is a dict: {"text": "...", "files": [...]}
    text_content = message.get("text") if isinstance(message, dict) else message
    docs=retrieve_context(text_content,faiss_index)
    answer =  generate_answer(text_content,docs)
    if text_content:
        return answer.content

demo = gr.ChatInterface(
    fn=respond,
    multimodal=False,
    chatbot=gr.Chatbot(value=[{"role": "assistant", "content": "Welcome! I am your AI assistant."}]), 
)
demo.launch()
#gr.ChatInterface(
#   fn=respond, 
#   examples=["Hello, I am a Alex, your personal HR assistant. How can I help you today?"],   
#).launch()


* Running on local URL:  http://127.0.0.1:7890
* To create a public link, set `share=True` in `launch()`.
